# E44 --- a inclinação da banda em toda escala

A tabela de horizontes do capítulo respondia "quanto a série é mais estreita que a previsão"
com um número por horizonte. Este caderno mede o número da série inteira: a inclinação do
passo médio do caminho contra o atraso, em log-log. É a leitura de Higuchi --- a inclinação
como um número repetível que caracteriza a série, lida do outro lado como dimensão.

**O que se mede.**

1. o passo médio do caminho em cada atraso e a inclinação (o gamma), com o MESMO estimador
   e os MESMOS atrasos em três lugares: os mundos do passeio puro, a série real e a série
   baralhada --- o controle de ordem herdado do E22;
2. a dimensão do caminho, dois menos o gamma;
3. o gamma por faixa de atraso, porque um número só pode não descrever a série.

In [1]:
# <- brinque com: SERIE, ATRASOS, JANELAS_MINIMAS, MUNDOS, SORTEIOS, SEMENTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, graficos, nivel, volatilidade

RAIZ = Path.cwd()
SERIE = "sp500.csv"       # a série do arquivo do projeto anterior (.old/dados/)
ATRASOS = (1, 2, 4, 8, 16, 32, 64, 128, 256, 512)  # os atrasos da reta, em progressão de dois
JANELAS_MINIMAS = 30      # atraso com menos janelas que isso fica de fora do ajuste
MUNDOS = 64               # mundos do passeio puro
SORTEIOS = 200            # baralhamentos do controle de ordem (herdado do E22)
FAIXAS = {"curta": (1, 2, 4, 8), "media": (16, 32, 64), "longa": (128, 256, 512)}
SEMENTE = 111

precos = dados.carregar_serie(SERIE)
retornos = volatilidade.retornos_log(precos).to_numpy()
desvio_diario = float(retornos.std(ddof=1))
print("frevolab %s | %s: %d pregões | desvio de um dia: %.5f" % (
    frevolab.VERSAO, SERIE, len(precos), desvio_diario))
print("atrasos que entram no ajuste:", [k for k in ATRASOS
                                        if len(precos) - k >= JANELAS_MINIMAS])

frevolab 0.1.0 | sp500.csv: 6719 pregões | desvio de um dia: 0.01213
atrasos que entram no ajuste: [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]


## O passeio puro: a régua de inclinação meio

Nos mundos do passeio o passo médio cresce com a raiz do atraso --- a proposição do
capítulo, agora lida como reta. A inclinação esperada é meio; a dimensão, três meios.

In [2]:
# Os MUNDOS passeios puros: mesmo desvio do dia, mesmo comprimento da série.
# A conta, o estimador e os atrasos são os mesmos da biblioteca em todos os lados da
# comparação --- a inclinação não pode divergir por construção diferente.
sorteio = np.random.default_rng(SEMENTE)
caminhos = np.exp(sorteio.normal(0.0, desvio_diario, (MUNDOS, len(precos) - 1)).cumsum(axis=1))
gammas_mundos = [nivel.expoente_do_caminho(c, ATRASOS, JANELAS_MINIMAS) for c in caminhos]
passo_mundos = {k: float(np.median([nivel.passo_do_caminho(c, (k,))[k] for c in caminhos]))
                for k in ATRASOS}
print("gamma dos mundos: média %.4f | dispersão %.4f" % (
    float(np.mean(gammas_mundos)), float(np.std(gammas_mundos, ddof=1))))
print("dimensão dos mundos: %.4f" % (2.0 - float(np.mean(gammas_mundos))))

gamma dos mundos: média 0.4998 | dispersão 0.0202
dimensão dos mundos: 1.5002


## A série real, e o controle de ordem

O mesmo estimador na série --- e nos baralhamentos dela: os mesmos dias, outra ordem. Se a
inclinação global fosse obra da ordem, o baralhado a destruiria; se fosse obra dos tamanhos
dos dias, o baralhado a reproduz.

In [3]:
# A série, o baralhado e o gamma por faixa.
gamma_real = nivel.expoente_do_caminho(precos.to_numpy(), ATRASOS, JANELAS_MINIMAS)
passo_real = nivel.passo_do_caminho(precos.to_numpy(), ATRASOS)
dimensao_real = nivel.dimensao_do_caminho(precos.to_numpy(), ATRASOS, JANELAS_MINIMAS)

controle = nivel.expoente_do_baralhado(retornos, ATRASOS, SORTEIOS,
                                       np.random.default_rng(SEMENTE + 1), JANELAS_MINIMAS)
print("gamma da série: %.4f | dimensão: %.4f" % (gamma_real, dimensao_real))
print("gamma do baralhado: média %.4f | dispersão %.4f | quantos %d" % (
    controle["media"], controle["dispersao"], controle["quantos"]))

faixa_real = {nome: nivel.expoente_do_caminho(precos.to_numpy(), fs, JANELAS_MINIMAS)
              for nome, fs in FAIXAS.items()}
faixa_mundos = {nome: float(np.mean([nivel.expoente_do_caminho(c, fs, JANELAS_MINIMAS)
                                     for c in caminhos]))
                for nome, fs in FAIXAS.items()}
faixa_baralhado = {}
for i, (nome, fs) in enumerate(FAIXAS.items()):
    faixa_baralhado[nome] = nivel.expoente_do_baralhado(
        retornos, fs, SORTEIOS, np.random.default_rng(SEMENTE + 2 + i),
        JANELAS_MINIMAS)["media"]
tabela_faixas = pd.DataFrame({"a série": faixa_real, "os mundos": faixa_mundos,
                              "o baralhado": faixa_baralhado})
print()
print(tabela_faixas.round(4).to_string())

gamma da série: 0.5342 | dimensão: 1.4658
gamma do baralhado: média 0.5322 | dispersão 0.0204 | quantos 200



       a série  os mundos  o baralhado
curta   0.4732     0.5006       0.5690
media   0.5079     0.4983       0.5178
longa   0.6780     0.4927       0.5341


## As figuras

In [4]:
# Figura 1: o passo médio contra o atraso, em log-log --- as três curvas e a régua.
regua = {k: desvio_diario * np.sqrt(2.0 / np.pi) * np.sqrt(k) for k in ATRASOS}

fig, eixo = plt.subplots(figsize=(8.6, 4.4))
eixo.loglog(ATRASOS, [100 * passo_real[k] for k in ATRASOS], "o-", color="#b03a2e",
            label="a série real")
eixo.loglog(ATRASOS, [100 * passo_mundos[k] for k in ATRASOS], "s--", color="#1f4e79",
            label="a mediana dos %d mundos do passeio" % MUNDOS)
eixo.loglog(ATRASOS, [100 * controle["passo_mediana"][k] for k in ATRASOS], "^-",
            color="#c78f2c", label="a mediana dos %d baralhamentos" % SORTEIOS)
eixo.loglog(list(ATRASOS), [100 * regua[k] for k in ATRASOS], color="#666666", lw=1.0,
            ls="-.", label="a régua de inclinação meio")
eixo.set_xlabel("atraso entre os dois dias (pregões)")
eixo.set_ylabel("passo médio do caminho (%)")
eixo.set_title("O passo médio em toda escala, nas três leituras")
eixo.grid(True, which="both", ls=":", lw=0.6, alpha=0.6)
eixo.legend(fontsize=8)
graficos.salvar(fig, "E44_expoente", 1)
plt.close(fig)

In [5]:
# Figura 2: a inclinação de cada mundo e de cada baralhamento, com a série no meio.
fig, eixo = plt.subplots(figsize=(8.6, 4.2))
borda = 0.012
fachos = np.linspace(min(min(gammas_mundos), min(controle["gammas"])) - borda,
                     max(max(gammas_mundos), max(controle["gammas"])) + borda, 48)
eixo.hist(gammas_mundos, bins=fachos, color="#1f4e79", alpha=0.55,
          label="o gamma dos %d mundos" % MUNDOS)
eixo.hist(controle["gammas"], bins=fachos, color="#c78f2c", alpha=0.55,
          label="o gamma dos %d baralhamentos" % SORTEIOS)
eixo.axvline(gamma_real, color="#b03a2e", lw=1.6,
             label="o gamma da série: %.3f" % gamma_real)
eixo.axvline(0.5, color="#666666", ls=":", lw=1.2, label="meio, o gamma do passeio")
eixo.set_xlabel("gamma, a inclinação do caminho")
eixo.set_ylabel("contagem")
eixo.set_title("Onde caem as inclinações dos mundos, dos baralhamentos e da série")
eixo.legend(fontsize=8)
graficos.salvar(fig, "E44_expoente", 2)
plt.close(fig)

In [6]:
# Leitura visual das figuras (declaratória, contra o PNG):
# 1) na figura 1, a série nasce junto do baralhado no atraso de um dia (os mesmos dias),
#    corre ABAIXO da mediana dos mundos nos atrasos curtos e médios e a alcança no maior;
#    o baralhado sobe mais rápido que a série e cruza a linha dos mundos antes dela; a
#    régua de inclinação meio cai junto com a mediana dos mundos;
# 2) na figura 2, a pilha dos mundos se acumula no meio, a do baralhado um pouco à direita
#    dela, e a linha da série cai DENTRO da pilha do baralhado --- o expoente global é
#    número dos tamanhos dos dias, e a ordem não o muda.

In [7]:
# O resultado: um objeto por grandeza, em português, para o livro citar por comando.
contagem = {"expoente_atrasos": len(ATRASOS),
            "expoente_atraso_maximo": max(ATRASOS),
            "expoente_janelas_minimas": JANELAS_MINIMAS,
            "expoente_mundos": MUNDOS,
            "expoente_sorteios": SORTEIOS}
medidas = {k: round(float(v), 4) for k, v in {
    "expoente_gamma_passeio": float(np.mean(gammas_mundos)),
    "expoente_gamma_passeio_dispersao": float(np.std(gammas_mundos, ddof=1)),
    "expoente_dimensao_passeio": 2.0 - float(np.mean(gammas_mundos)),
    "expoente_gamma_real": gamma_real,
    "expoente_dimensao_real": dimensao_real,
    "expoente_gamma_baralhado_media": controle["media"],
    "expoente_gamma_baralhado_dispersao": controle["dispersao"],
    "expoente_dimensao_baralhado_media": 2.0 - float(controle["media"]),
    "expoente_gamma_faixa_curta_real": faixa_real["curta"],
    "expoente_gamma_faixa_media_real": faixa_real["media"],
    "expoente_gamma_faixa_longa_real": faixa_real["longa"],
    "expoente_gamma_faixa_curta_passeio": faixa_mundos["curta"],
    "expoente_gamma_faixa_media_passeio": faixa_mundos["media"],
    "expoente_gamma_faixa_longa_passeio": faixa_mundos["longa"],
    "expoente_gamma_faixa_curta_baralhado": faixa_baralhado["curta"],
    "expoente_gamma_faixa_media_baralhado": faixa_baralhado["media"],
    "expoente_gamma_faixa_longa_baralhado": faixa_baralhado["longa"],
}.items()}
resultado = {**contagem, **medidas}

caminho = Path("lab/resultados/E44_expoente.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True),
                   encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))
print(json.dumps(resultado, ensure_ascii=False, indent=1, sort_keys=True))

lab/resultados/E44_expoente.json gravado | 22 grandezas
{
 "expoente_atraso_maximo": 512,
 "expoente_atrasos": 10,
 "expoente_dimensao_baralhado_media": 1.4678,
 "expoente_dimensao_passeio": 1.5002,
 "expoente_dimensao_real": 1.4658,
 "expoente_gamma_baralhado_dispersao": 0.0204,
 "expoente_gamma_baralhado_media": 0.5322,
 "expoente_gamma_faixa_curta_baralhado": 0.569,
 "expoente_gamma_faixa_curta_passeio": 0.5006,
 "expoente_gamma_faixa_curta_real": 0.4732,
 "expoente_gamma_faixa_longa_baralhado": 0.5341,
 "expoente_gamma_faixa_longa_passeio": 0.4927,
 "expoente_gamma_faixa_longa_real": 0.678,
 "expoente_gamma_faixa_media_baralhado": 0.5178,
 "expoente_gamma_faixa_media_passeio": 0.4983,
 "expoente_gamma_faixa_media_real": 0.5079,
 "expoente_gamma_passeio": 0.4998,
 "expoente_gamma_passeio_dispersao": 0.0202,
 "expoente_gamma_real": 0.5342,
 "expoente_janelas_minimas": 30,
 "expoente_mundos": 64,
 "expoente_sorteios": 200
}
